<a href="https://colab.research.google.com/github/sebabecerra/Econometria-II-DEN/blob/main/Tarea2/Tarea2_Twins_Bootstrap_Solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/sebabecerra/Econometria-II-DEN/blob/main/Tarea2/Tarea2_Twins_Bootstrap_Solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea 2: MCO, diferencias entre gemelos y bootstrap

## Econometría II

Para esta tarea utilice la base de datos TWINSAK, basada en el estudio de gemelos de Ashenfelter y Krueger. La base contiene, entre otras variables, el logaritmo del salario semanal (lwage1, lwage2), años de educación (educ1, educ2), edad y género.

El informe deberá abordar, de manera breve y ordenada, los siguientes aspectos:

1. Trate inicialmente a ambos miembros de cada par de gemelos como observaciones individuales y estime:

   $$
   lwage_i = \beta_0 + \beta_1 educ_i + \beta_2 age_i + \beta_3 age_i^2 + \varepsilon_i.
   $$

   Interprete el coeficiente de educación y discuta brevemente si tratar a los gemelos como observaciones independientes es un supuesto razonable.

2. Estime luego el modelo en diferencias dentro de cada par:

   $$
   \Delta lwage_j = \beta_1 \Delta educ_j + \Delta \varepsilon_j.
   $$

   Explique qué características no observadas comunes a ambos gemelos se eliminan al tomar diferencias y por qué esta estrategia puede reducir algunos problemas de variable omitida.

3. Para el modelo en diferencias, estime el error estándar de $\hat\beta_1$ utilizando:

   (i) MCO convencional;

   (ii) errores estándar robustos HC1;

   (iii) bootstrap paramétrico;

   (iv) bootstrap no paramétrico de pares.

   Utilice al menos $R = 9,999$ réplicas bootstrap y la misma semilla para ambos procedimientos. Presente los resultados en una única tabla y compare los errores estándar e intervalos de confianza al 95 %.

4. Explique con precisión qué se remuestrea en el bootstrap paramétrico y en el bootstrap de pares. ¿Qué supuestos adicionales impone el bootstrap paramétrico?

5. Suponga ahora que en la regresión en diferencias se ha omitido la diferencia en capacidad entre los gemelos. Suponga que la capacidad puede aproximarse mediante IQ y que el efecto parcial de IQ sobre el logaritmo del salario, controlando por educación, es aproximadamente 0,0063.

   Utilizando la fórmula de sesgo por variable omitida, explique:

   (a) qué información adicional se necesita para determinar el signo y magnitud del sesgo en el coeficiente de educación;

   (b) bajo qué relación entre $\Delta educ$ y $\Delta IQ$ el coeficiente de educación estaría sesgado hacia arriba o hacia abajo;

   (c) si el bootstrap podría corregir este problema de endogeneidad.

6. Compare conceptualmente el modelo en niveles y el modelo en diferencias. ¿Puede interpretarse alguno de los dos coeficientes de educación como causal? Fundamente brevemente su respuesta.

**Extensión máxima:** una página, sin incluir la tabla ni las referencias. Adjunte el archivo `.do` utilizado para obtener los resultados.

# Solución

**Modelos principales**

En niveles se estima

$$
lwage_i = \beta_0 + \beta_1 educ_i + \beta_2 age_i + \beta_3 age_i^2 + \varepsilon_i.
$$

En diferencias dentro de cada pareja $j$ se estima, **sin constante**, la ecuación indicada en el enunciado:

$$
\Delta lwage_j = \beta_1\Delta educ_j + \Delta\varepsilon_j,
$$

donde $\Delta lwage_j=lwage_{1j}-lwage_{2j}$ y $\Delta educ_j=educ_{1j}-educ_{2j}$. Cambiar el orden de los gemelos multiplica ambas diferencias por $-1$ y no altera $\hat\beta_1$.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from IPython.display import display

pd.set_option('display.float_format', lambda x: f'{x:,.5f}')

# Base alojada públicamente en GitHub.
URL_data = 'https://raw.githubusercontent.com/sebabecerra/Econometria-II-DEN/main/Tarea2/twinsak.dta'
data = pd.read_stata(URL_data)

print(data.head())
print(data.describe())
print(data['male1'].value_counts())

       age  educ1  educ2  lwage1  lwage2  male1
0 33.25120     16     16 2.16102 2.42037      0
1 43.57016     12     19 2.16905 2.89037      0
2 30.96783     12     12 2.79178 2.80336      1
3 34.63381     14     14 2.82435 2.26337      1
4 34.97878     15     13 2.03209 3.55535      0
            age     educ1     educ2    lwage1    lwage2     male1
count 148.00000 148.00000 148.00000 148.00000 148.00000 148.00000
mean   36.59916  14.16216  14.10135   2.34694   2.41198   0.45946
std    10.40458   2.13189   2.14679   0.63295   0.60790   0.50005
min    18.78166  10.00000  10.00000   0.51083   0.57982   0.00000
25%    28.97467  12.00000  12.00000   1.94352   1.94591   0.00000
50%    35.40452  14.00000  14.00000   2.35941   2.39699   0.00000
75%    43.06503  16.00000  16.00000   2.73613   2.77259   1.00000
max    64.13963  20.00000  20.00000   4.60517   4.11966   1.00000
male1
0    80
1    68
Name: count, dtype: int64


## Pregunta 1 — Modelo en niveles

Se apilan los dos miembros de cada pareja, por lo que la muestra pasa de 148 pares a 296 observaciones individuales. El coeficiente de educación es una **semielasticidad**: manteniendo fija la edad, un año adicional de educación se asocia con aproximadamente $100\hat\beta_1\%$ más salario semanal; el cambio porcentual exacto es $100(e^{\hat\beta_1}-1)$.

El MCO convencional trata las 296 filas como independientes. Como dos gemelos comparten genes, edad, hogar de origen y parte importante de su ambiente, sus errores probablemente están correlacionados. Por eso se presenta también, como diagnóstico, el error estándar agrupado por pareja. Este ajuste mejora la inferencia ante correlación intrapareja, pero **no elimina** sesgo por variables omitidas correlacionadas con educación.

In [ ]:
base_long = (
    pd.wide_to_long(
        data.reset_index(names="id"),
        stubnames=["educ", "lwage"],
        i="id",
        j="gemelo"
    )
    .reset_index()
)
base_long['age2'] = base_long['age'] ** 2
base_long = base_long[['id', 'gemelo', 'male1', 'lwage', 'educ', 'age', 'age2']]
base_long = base_long.sort_values(by=['id', 'gemelo'], ascending=True)
display(base_long.head())

,id,gemelo,male1,lwage,educ,age,age2
0,0,1,0,2.16102,16,33.25120,"1,105.64246"
148,0,2,0,2.42037,16,33.25120,"1,105.64246"
1,1,1,0,2.16905,12,43.57016,"1,898.35889"
149,1,2,0,2.89037,19,43.57016,"1,898.35889"
2,2,1,1,2.79178,12,30.96783,959.00653


In [ ]:
X = sm.add_constant(base_long[['educ', 'age', 'age2']])
y = base_long['lwage']
modelo = sm.OLS(y, X).fit()
# Resumen del modelo OLS estándar
print('Resumen del Modelo OLS Estándar:')
print(modelo.summary())

In [ ]:
b = modelo.params

base_long["lwage_hat"] = (
    b["const"]
    + b["educ"] * base_long["educ"]
    + b["age"] * base_long["age"]
    + b["age2"] * base_long["age2"]
)

base_long["residuo"] = (
    base_long["lwage"]
    - base_long["lwage_hat"]
)

residuos_pares = base_long.pivot(
    index="id",
    columns="gemelo",
    values="residuo"
)

u1 = residuos_pares[1].to_numpy()
u2 = residuos_pares[2].to_numpy()

rho_intra = np.sum(u1 * u2) / (
    0.5 * np.sum(u1**2 + u2**2)
)

print(f"Correlación residual intrapareja: {rho_intra:.3f}")

Correlación residual intrapareja: 0.496


No es completamente razonable tratar a los gemelos como observaciones independientes. Aunque cada gemelo es una persona distinta, los integrantes de una pareja comparten genética, familia, edad y ambiente. Estos factores comunes pueden quedar en el error de la regresión, generando correlación entre los errores de ambos:

$$
\operatorname{Cov}(\varepsilon_{1g},\varepsilon_{2g}\mid X)\neq 0.
$$

Ignorar esta dependencia no cambia el coeficiente MCO, pero puede subestimar su error estándar y exagerar la precisión. En efecto, el error estándar de educación aumenta de 0,0150 con MCO a 0,0176 al agrupar por pareja. Por ello, es más apropiado considerar independientes a las parejas, pero permitir correlación entre los gemelos de una misma pareja mediante errores estándar agrupados.

In [ ]:
Xm = X.to_numpy()
u  = modelo.resid.to_numpy()
N, K = Xm.shape
G = base_long['id'].nunique()

In [ ]:
G

148

**Resultado.** El coeficiente de educación es $\hat\beta_1=0{,}0941$: a edad constante, un año adicional de educación se asocia con cerca de **9,4%** más salario semanal (9,9% usando la transformación exacta). El error estándar MCO es 0,0150 y aumenta a 0,0176 al agrupar por pareja, evidencia de que tratar a los gemelos como independientes afecta la incertidumbre estimada.

La asociación no es necesariamente causal: capacidad, preferencias, ambiente familiar y otros determinantes de educación y salarios pueden permanecer en el error.

### Error estándar agrupado por pareja

Agrupar por pareja no modifica el estimador MCO:

$$
\hat{\beta}=(X'X)^{-1}X'y.
$$

Solamente cambia la estimación de su varianza. Para permitir que los errores de los dos gemelos estén correlacionados, primero se suman sus contribuciones dentro de cada pareja:

$$
\sum_{i\in g}x_{ig}\hat{u}_{ig}.
$$

La matriz de varianza agrupada se calcula como:

$$
\widehat{\operatorname{Var}}_{\mathrm{cluster}}(\hat{\beta})
=
c(X'X)^{-1}
\left[
\sum_{g=1}^{G}
\left(
\sum_{i\in g}x_{ig}\hat{u}_{ig}
\right)
\left(
\sum_{i\in g}x_{ig}\hat{u}_{ig}
\right)'
\right]
(X'X)^{-1},
$$

donde la corrección por muestra finita es:

$$
c=
\frac{G}{G-1}
\frac{N-1}{N-K}.
$$

En esta aplicación:

$$
N=296,
\qquad
K=4,
\qquad
G=148.
$$

Al multiplicar las contribuciones agrupadas aparecen productos entre los residuos de ambos gemelos, como:

$$
\hat{u}_{1g}\hat{u}_{2g}.
$$

Estos términos permiten incorporar la correlación de los errores dentro de cada pareja. Finalmente, los errores estándar se obtienen tomando la raíz cuadrada de la diagonal:

$$
SE_{\mathrm{cluster}}(\hat{\beta}_j)
=
\sqrt{
\left[
\widehat{\operatorname{Var}}_{\mathrm{cluster}}(\hat{\beta})
\right]_{jj}
}.
$$

In [ ]:
# Estimar el modelo y obtener los residuos
u = modelo.resid

# Elementos comunes
N, K = X.shape
XX_inv = np.linalg.inv(X.T @ X)
xu = X.mul(u, axis=0)

# HC1 manual
centro_HC1 = xu.T @ xu
V_HC1 = (N / (N - K)) * XX_inv @ centro_HC1 @ XX_inv
SE_HC1 = np.sqrt(np.diag(V_HC1))

# Errores estándar agrupados por pareja
G = base_long["id"].nunique()

xu_pareja = xu.groupby(base_long["id"]).sum()
centro_cluster = xu_pareja.T @ xu_pareja

correccion = (G / (G - 1)) * ((N - 1) / (N - K))

V_cluster = correccion * XX_inv @ centro_cluster @ XX_inv
SE_cluster = np.sqrt(np.diag(V_cluster))

# Comparación final
pd.DataFrame({
    "Coeficiente": modelo.params,
    "SE MCO": modelo.bse,
    "SE HC1": SE_HC1,
    "SE agrupado por pareja": SE_cluster
}, index=X.columns).round(4)

,Coeficiente,SE MCO,SE HC1,SE agrupado por pareja
const,-0.71500,0.42580,0.49650,0.55410
educ,0.09410,0.01500,0.01530,0.01760
age,0.07680,0.01930,0.02110,0.02370
age2,-0.00070,0.00020,0.00030,0.00030


## Pregunta 2 — Modelo en diferencias dentro de cada par

Suponga que el error individual puede escribirse como

$$
\varepsilon_{ij}=a_j+v_{ij},
$$

donde $a_j$ contiene características comunes a la pareja. Al restar las ecuaciones de los dos gemelos,

$$
\varepsilon_{1j}-\varepsilon_{2j}=(a_j+v_{1j})-(a_j+v_{2j})=v_{1j}-v_{2j},
$$

se eliminan exactamente los factores compartidos e invariantes dentro del par: edad y cohorte, antecedentes familiares, educación de los padres, vecindario de origen y, en gemelos idénticos, gran parte de la dotación genética.

Esto reduce el sesgo por variables omitidas **comunes**, pero no elimina diferencias individuales de capacidad, motivación, salud o redes. Además, la diferenciación puede agravar el sesgo por error de medición en educación, porque elimina señal común y deja relativamente más ruido.

In [23]:
data['d_lwage'] = data['lwage1'] - data['lwage2']
data['d_educ'] = data['educ1'] - data['educ2']

# Matriz de diseño sin constante, de acuerdo con el enunciado.
X_dif = data[['d_educ']]
modelo_dif = sm.OLS(data['d_lwage'], X_dif).fit()

display(pd.DataFrame({
    'Coeficiente': modelo_dif.params,
    'SE MCO': modelo_dif.bse,
    't': modelo_dif.tvalues,
    'p-valor': modelo_dif.pvalues,
}))

print(f'Pares: {len(data)}')
print(f'Pares con distinta educación: {(data["d_educ"] != 0).sum()}')
print(f'Retorno aproximado: {100*modelo_dif.params["d_educ"]:.2f}%')
print(f'Retorno porcentual exacto: {100*np.expm1(modelo_dif.params["d_educ"]):.2f}%')

,Coeficiente,SE MCO,t,p-valor
d_educ,0.10672,0.02459,4.33959,0.00003


Pares: 148
Pares con distinta educación: 75
R² sin centrar = 0.1136
Retorno aproximado: 10.67%
Retorno porcentual exacto: 11.26%


In [25]:
print(modelo_dif.summary())

                                 OLS Regression Results                                
Dep. Variable:                d_lwage   R-squared (uncentered):                   0.114
Model:                            OLS   Adj. R-squared (uncentered):              0.108
Method:                 Least Squares   F-statistic:                              18.83
Date:                Mon, 17 Aug 2026   Prob (F-statistic):                    2.64e-05
Time:                        01:39:30   Log-Likelihood:                         -120.85
No. Observations:                 148   AIC:                                      243.7
Df Residuals:                     147   BIC:                                      246.7
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

**Resultado.** En diferencias, $\hat\beta_1=0{,}1067$: dentro de una pareja, el gemelo con un año adicional de educación tiene, en promedio, cerca de **10,7%** más salario semanal (11,3% con la transformación exacta). La identificación proviene de los pares con $\Delta educ\neq 0$; los pares con igual educación no aportan al numerador ni al denominador de la pendiente, aunque sí entran al cálculo de residuos.

## Pregunta 3 — Cuatro estimadores del error estándar

Para la regresión sin constante,

$$
\hat\beta_1=\frac{\sum_j x_jy_j}{\sum_jx_j^2},\qquad x_j=\Delta educ_j,\; y_j=\Delta lwage_j.
$$

Se comparan:

1. **MCO convencional:** $\widehat{Var}(\hat\beta_1)=\hat\sigma^2/(\sum_jx_j^2)$, con $\hat\sigma^2=\sum_j\hat u_j^2/(n-1)$.
2. **HC1:** sándwich robusto a heterocedasticidad, con corrección $n/(n-1)$.
3. **Bootstrap paramétrico:** condiciona en $x$ y genera errores normales con varianza estimada.
4. **Bootstrap de pares:** remuestrea filas completas $(x_j,y_j)$ con reemplazo.

Se usan $R=9.999$ réplicas y la semilla `20260815` en ambos procedimientos. Los intervalos MCO y HC1 usan el cuantil $t_{n-1}$; los intervalos bootstrap son percentiles 2,5% y 97,5% de las pendientes simuladas.

In [26]:
x = data['d_educ'].to_numpy(dtype=float)
y = data['d_lwage'].to_numpy(dtype=float)
n = len(x)
k = 1

beta_hat = float(modelo_dif.params['d_educ'])
residuos = y - beta_hat*x
sigma2_hat = residuos @ residuos / (n-k)

# MCO convencional y HC1
se_mco = float(modelo_dif.bse['d_educ'])
modelo_dif_hc1 = modelo_dif.get_robustcov_results(cov_type='HC1')
se_hc1 = float(modelo_dif_hc1.bse[0])
tcrit = stats.t.ppf(0.975, n-k)

print(f'β̂1 = {beta_hat:.6f}')
print(f'σ̂² = {sigma2_hat:.6f}')
print(f'SE MCO = {se_mco:.6f}')
print(f'SE HC1 = {se_hc1:.6f}')

β̂1 = 0.106725
σ̂² = 0.301809
SE MCO = 0.024593
SE HC1 = 0.026201


In [27]:
R = 9_999
SEMILLA = 20_260_815

# (iii) Bootstrap paramétrico: X fijo, u* ~ N(0, sigma² estimada)
rng_param = np.random.default_rng(SEMILLA)
errores_star = rng_param.normal(0, np.sqrt(sigma2_hat), size=(R, n))
y_star = beta_hat*x[None, :] + errores_star
beta_param = (y_star @ x) / (x @ x)

# (iv) Bootstrap no paramétrico de pares: remuestrea filas (x_j, y_j)
rng_pares = np.random.default_rng(SEMILLA)
indices = rng_pares.integers(0, n, size=(R, n))
x_star = x[indices]
y_star_pares = y[indices]
denominadores = np.sum(x_star**2, axis=1)

if np.any(denominadores == 0):
    raise RuntimeError('Alguna réplica no contiene variación en Δeduc.')

beta_pares = np.sum(x_star*y_star_pares, axis=1) / denominadores

print(f'Réplicas paramétricas: {len(beta_param):,}')
print(f'Réplicas de pares: {len(beta_pares):,}')

Réplicas paramétricas: 9,999
Réplicas de pares: 9,999


In [28]:
ic_mco = beta_hat + np.array([-1, 1])*tcrit*se_mco
ic_hc1 = beta_hat + np.array([-1, 1])*tcrit*se_hc1

tabla_inferencia = pd.DataFrame({
    'Método': ['MCO convencional', 'HC1', 'Bootstrap paramétrico', 'Bootstrap de pares'],
    'Coeficiente': [beta_hat]*4,
    'Error estándar': [
        se_mco,
        se_hc1,
        beta_param.std(ddof=1),
        beta_pares.std(ddof=1),
    ],
    'IC 95% inferior': [
        ic_mco[0],
        ic_hc1[0],
        np.quantile(beta_param, 0.025),
        np.quantile(beta_pares, 0.025),
    ],
    'IC 95% superior': [
        ic_mco[1],
        ic_hc1[1],
        np.quantile(beta_param, 0.975),
        np.quantile(beta_pares, 0.975),
    ],
}).set_index('Método')

display(tabla_inferencia)

print('Media bootstrap paramétrico:', f'{beta_param.mean():.6f}')
print('Media bootstrap de pares:', f'{beta_pares.mean():.6f}')

,Coeficiente,Error estándar,IC 95% inferior,IC 95% superior
Método,,,,
MCO convencional,0.10672,0.02459,0.05812,0.15533
HC1,0.10672,0.02620,0.05494,0.15850
Bootstrap paramétrico,0.10672,0.02428,0.05968,0.15513
Bootstrap de pares,0.10672,0.02647,0.05899,0.16257


Media bootstrap paramétrico: 0.106694
Media bootstrap de pares: 0.108399


**Comparación.** Los cuatro procedimientos sitúan el error estándar entre 0,0243 y 0,0265. El bootstrap paramétrico es el más cercano al MCO convencional porque reproduce explícitamente su supuesto homocedástico-normal. HC1 y el bootstrap de pares son algo más conservadores, pues permiten que la dispersión de $\Delta lwage$ varíe con $\Delta educ$ y que la distribución empírica sea no normal.

Todos los intervalos excluyen cero, por lo que la asociación dentro de pareja es estadísticamente distinta de cero al 5%. Esto es evidencia sobre precisión muestral **condicional al modelo**, no una prueba de causalidad.

## Pregunta 4 — ¿Qué se remuestrea?

### Bootstrap paramétrico

Se mantienen fijos los 148 valores observados de $x_j=\Delta educ_j$. Para cada réplica $r$ se generan

$$
u_j^{*(r)}\overset{iid}{\sim}N(0,\hat\sigma^2),\qquad
y_j^{*(r)}=\hat\beta_1x_j+u_j^{*(r)},
$$

y se reestima la pendiente. Además de la correcta especificación lineal y la exogeneidad, este método impone errores iid, homocedásticos y normales, y trata la distribución de $x$ como fija. Si esos supuestos son incorrectos, la distribución simulada puede representar mal la incertidumbre real.

### Bootstrap no paramétrico de pares

Se extraen con reemplazo 148 filas completas $(\Delta educ_j,\Delta lwage_j)$ de la muestra empírica y se reestima la pendiente. Se preserva la relación conjunta observada entre educación y salario dentro de cada fila, incluida la heterocedasticidad. No impone normalidad ni una forma paramétrica para los errores, pero requiere que los **pares de gemelos** sean independientes y representativos entre sí. La unidad de remuestreo es la pareja, nunca cada gemelo por separado.

## Pregunta 5 — Sesgo por omitir la diferencia en capacidad

Suponga que el modelo verdadero es

$$
\Delta lwage_j=\beta_1\Delta educ_j+\gamma\Delta IQ_j+v_j,
\qquad \gamma\approx 0{,}0063.
$$

Si se omite $\Delta IQ$, la fórmula de sesgo por variable omitida es

$$
\operatorname{plim}(\tilde\beta_1)-\beta_1
=\gamma\frac{\operatorname{Cov}(\Delta educ,\Delta IQ)}
{\operatorname{Var}(\Delta educ)}
=0{,}0063\,\delta,
$$

donde $\delta$ es la pendiente de la regresión auxiliar de $\Delta IQ$ sobre $\Delta educ$ (sin constante, coherente con la ecuación diferenciada).

**(a)** Para conocer signo y magnitud se necesita observar $\Delta IQ$, o al menos conocer $\operatorname{Cov}(\Delta educ,\Delta IQ)$ y $\operatorname{Var}(\Delta educ)$; equivalentemente, se necesita $\delta$. La base entregada no contiene IQ, por lo que el sesgo no puede cuantificarse directamente.

**(b)** Como $\gamma=0{,}0063>0$, si $\Delta educ$ y $\Delta IQ$ se relacionan positivamente ($\delta>0$), el coeficiente de educación está sesgado **hacia arriba**. Si la relación es negativa ($\delta<0$), está sesgado **hacia abajo**. Si son ortogonales, no hay sesgo por esta omisión.

**(c)** Ninguno de los bootstraps corrige endogeneidad. Ambos reproducen el estimador aplicado a datos generados o remuestreados bajo la misma relación contaminada. Pueden aproximar su distribución y error estándar, pero la distribución quedará centrada alrededor del coeficiente sesgado. Corregirlo exige controlar por $\Delta IQ$, usar una variable instrumental válida o adoptar otro diseño de identificación.

In [ ]:
# Sensibilidad: sesgo implícito para valores hipotéticos de delta.
delta_hipotetico = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])

tabla_sensibilidad = pd.DataFrame({
    'δ: efecto de Δeduc sobre ΔIQ': delta_hipotetico,
    'Sesgo = 0.0063·δ': 0.0063*delta_hipotetico,
    'β causal implícito = β observado − sesgo': beta_hat - 0.0063*delta_hipotetico,
})

display(tabla_sensibilidad)

## Pregunta 6 — Niveles, diferencias y causalidad

| Aspecto | Modelo en niveles | Modelo en diferencias |
|---|---|---|
| Variación utilizada | Entre todos los individuos | Dentro de cada pareja |
| Factores familiares/genéticos comunes | Permanecen en el error | Se eliminan si son comunes e invariantes |
| Dependencia intrapareja | Debe corregirse en la inferencia | Cada pareja produce una observación |
| Riesgos principales | Selección educativa y variables omitidas | Capacidad individual, error de medición y poca variación dentro del par |

El coeficiente en niveles compara personas de distinta educación, edad y antecedentes no observados. Sin el supuesto fuerte $E[\varepsilon_i\mid educ_i,age_i]=0$, es una asociación, no un efecto causal.

El estimador en diferencias controla automáticamente toda característica compartida e invariante, por lo que ofrece una comparación más convincente. Sin embargo, solo es causal si

$$
E[\Delta\varepsilon_j\mid\Delta educ_j]=0,
$$

es decir, si las diferencias de educación no están correlacionadas con diferencias de capacidad, motivación, salud u otros determinantes salariales, y si la educación está medida sin error relevante. La condición no puede verificarse con esta base. Por tanto, **ninguno de los dos coeficientes puede declararse causal solo a partir de estas regresiones**; el de diferencias requiere menos supuestos sobre factores comunes, pero conserva posibles confusores individuales.

## Resumen de resultados

| Pregunta | Resultado principal |
|---|---|
| 1 | Niveles: $\hat\beta_{educ}=0{,}0941$; SE MCO 0,0150; SE agrupado 0,0176 |
| 2 | Diferencias: $\hat\beta_{educ}=0{,}1067$ |
| 3 | SE entre 0,0243 y 0,0265; todos los IC 95% excluyen cero |
| 4 | Paramétrico: errores normales con $X$ fijo; pares: filas $(\Delta educ,\Delta lwage)$ |
| 5 | Sesgo $=0{,}0063\,\delta$; el bootstrap no corrige endogeneidad |
| 6 | Diferencias elimina confusores comunes, pero causalidad exige exogeneidad dentro del par |

### Reproducibilidad

- Base: `twinsak.dta` (148 parejas).
- Réplicas: $R=9.999$ en ambos bootstraps.
- Semilla: `20260815`, reinicializada para cada procedimiento.
- Modelo en diferencias: sin constante, siguiendo exactamente el enunciado.

### Referencias breves

- Ashenfelter, O. y Krueger, A. (1994). *Estimates of the Economic Return to Schooling from a New Sample of Twins*.
- Efron, B. y Tibshirani, R. (1993). *An Introduction to the Bootstrap*.